# DACON 딥보이스 탐지: ASVspoof2019 20,000개 학습 → `best.pt` → `submit.zip`

이 노트북은 Google Colab GPU에서 위에서 아래로 실행하도록 작성되었습니다.

**구현 범위**

- Google Drive의 ASVspoof2019 LA 데이터에서 정확히 20,000개 사용
  - 공식 `train`: 16,000개
  - 공식 `dev`: 4,000개 검증
- `XLS-R-300M + attentive statistics pooling` 음성 딥페이크 탐지기 학습
- DACON과 동일한 방식의 EER로 검증하고 최저 EER 모델을 `best.pt`로 저장
- PANNs CNN14로 `VOICE_PRESENT_PROB`, `MUSIC_PRESENT_PROB` 생성
- MP3/WAV/FLAC, mono/stereo, 4초~60초, 전화채널 가능성을 고려한 추론 코드 생성
- DACON 규격의 `model/`, `script.py`, `requirements.txt`를 담은 `submit.zip` 생성

> **중요한 한계:** ASVspoof2019는 음성 real/fake 데이터만 포함합니다. 따라서 이 노트북은
> `FILE_FAKE_PROB`와 `VOICE_FAKE_PROB`를 제대로 학습하지만, `MUSIC_FAKE_PROB`를 직접
> 학습하지 못합니다. 음악 Fake에는 음성 탐지 점수를 연속값 fallback으로 사용합니다.
> 제출 형식은 완전하지만 상위권을 목표로 한다면 이후 FakeMusicCaps/SONICS 음악 branch를
> 반드시 추가하는 것이 좋습니다.

공식 조건: 1,200개 파일, 16 kHz, MP3/WAV/FLAC, 추론 60분, L4 22.4 GiB,
오프라인 실행, `output/submission.csv` 생성.

- [대회 평가 및 코드 제출 조건](https://dacon.io/competitions/official/236749/overview/evaluation)
- [데이터/제출 컬럼 설명](https://dacon.io/competitions/official/236749/data)
- [파일 단위 독립 예측 규칙](https://dacon.io/competitions/official/236749/overview/rules)

## 0. Colab 런타임 준비

Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 먼저 선택하세요.
PyTorch/torchaudio는 Colab의 CUDA 조합을 유지하고 나머지 패키지만 설치합니다.

In [ ]:
!nvidia-smi
!pip -q install \
    "transformers==4.57.6" \
    "accelerate==1.9.0" \
    "librosa==0.10.2.post1" \
    "soundfile==0.12.1" \
    "soxr==0.5.0.post1" \
    "panns-inference==0.1.1"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 설정

`ASV_ROOT`를 본인의 **압축 해제된** ASVspoof2019 폴더로 수정하세요.
폴더 아래에 protocol `.txt` 파일과 LA train/dev의 `.flac` 파일이 있어야 합니다.

Colab T4 기준 기본값은 XLS-R의 마지막 4개 Transformer layer만 학습합니다.
메모리가 부족하면 `BATCH_SIZE=1`, 시간이 부족하면 `EPOCHS=2`로 낮추세요.

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json
import os
import random
import shutil
import subprocess
import time
import zipfile

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio
import torchaudio.functional as AF
from sklearn.metrics import roc_curve
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    Wav2Vec2Config,
    Wav2Vec2Model,
    get_cosine_schedule_with_warmup,
)


@dataclass
class CFG:
    # 반드시 본인 Drive 경로로 수정
    ASV_ROOT: str = "/content/drive/MyDrive/datasets/ASVspoof2019_LA"
    OUTPUT_DIR: str = "/content/drive/MyDrive/dacon_deepvoice"

    MODEL_ID: str = "facebook/wav2vec2-xls-r-300m"
    N_TOTAL: int = 20_000
    N_VAL: int = 4_000
    SAMPLE_RATE: int = 16_000
    SEGMENT_SECONDS: float = 4.0
    MAX_SAMPLES: int = 64_000

    SEED: int = 42
    EPOCHS: int = 3
    BATCH_SIZE: int = 2
    GRAD_ACCUM_STEPS: int = 8
    NUM_WORKERS: int = 2
    UNFREEZE_LAST_N: int = 4
    LR_BACKBONE: float = 2e-6
    LR_HEAD: float = 2e-4
    WEIGHT_DECAY: float = 1e-4
    WARMUP_RATIO: float = 0.08
    MAX_GRAD_NORM: float = 1.0


cfg = CFG()
ASV_ROOT = Path(cfg.ASV_ROOT)
OUTPUT_DIR = Path(cfg.OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_PATH = OUTPUT_DIR / "best.pt"
MANIFEST_PATH = OUTPUT_DIR / "asvspoof_selected_20000.csv"

print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))
print("torch:", torch.__version__, "torchaudio:", torchaudio.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "GPU 런타임을 선택한 뒤 다시 실행하세요."
assert ASV_ROOT.exists(), f"ASV_ROOT를 확인하세요: {ASV_ROOT}"

## 2. ASVspoof2019 protocol과 오디오 인덱싱

protocol의 일반적인 한 줄은 다음과 같습니다.

```text
LA_0079 LA_T_1138215 - A01 spoof
```

폴더 구조가 조금 달라도 파일명 stem을 기준으로 오디오를 찾습니다.

In [ ]:
AUDIO_EXTENSIONS = {".flac", ".wav", ".mp3", ".ogg", ".m4a"}


def find_protocol(root: Path, split: str) -> Path:
    candidates = []
    for path in root.rglob("*.txt"):
        name = path.name.lower()
        score = 0
        if "protocol" in str(path).lower():
            score += 2
        if "cm" in name:
            score += 3
        if "la" in name:
            score += 2
        if f".{split}." in name or f"_{split}." in name or f"_{split}_" in name:
            score += 8
        if split in name:
            score += 2
        if score >= 10:
            candidates.append((score, path))
    if not candidates:
        raise FileNotFoundError(f"{split} protocol을 {root} 아래에서 찾지 못했습니다.")
    candidates.sort(key=lambda x: (-x[0], len(str(x[1]))))
    return candidates[0][1]


def build_audio_index(root: Path) -> dict[str, str]:
    index = {}
    for path in tqdm(root.rglob("*"), desc="오디오 인덱싱"):
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS:
            index[path.stem] = str(path)
    if not index:
        raise FileNotFoundError(f"오디오 파일을 찾지 못했습니다: {root}")
    return index


def read_protocol(protocol_path: Path, audio_index: dict[str, str], split: str) -> pd.DataFrame:
    rows = []
    with protocol_path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            speaker_id, utt_id, _, attack_id, label_text = parts[:5]
            label_text = label_text.lower()
            if label_text not in {"bonafide", "spoof"}:
                continue
            path = audio_index.get(utt_id)
            if path is None:
                continue
            rows.append({
                "speaker_id": speaker_id,
                "utt_id": utt_id,
                "attack_id": attack_id,
                "label": 0 if label_text == "bonafide" else 1,
                "split": split,
                "path": path,
            })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"protocol은 읽었지만 일치하는 오디오가 없습니다: {protocol_path}")
    return df


train_protocol = find_protocol(ASV_ROOT, "train")
dev_protocol = find_protocol(ASV_ROOT, "dev")
print("train protocol:", train_protocol)
print("dev protocol  :", dev_protocol)

audio_index = build_audio_index(ASV_ROOT)
train_all = read_protocol(train_protocol, audio_index, "train")
dev_all = read_protocol(dev_protocol, audio_index, "dev")

print("train 전체:\n", train_all["label"].value_counts().sort_index())
print("dev 전체:\n", dev_all["label"].value_counts().sort_index())

## 3. 정확히 20,000개 선택

- 학습 16,000개: train의 bonafide를 가능한 한 모두 보존하고 spoof로 나머지를 채움
- 검증 4,000개: dev에서 bonafide/spoof를 2,000개씩 균형 선택
- 학습 batch는 `WeightedRandomSampler`로 두 클래스를 균형화

In [ ]:
def sample_rows(df: pd.DataFrame, n: int, seed: int, prefer_balanced: bool) -> pd.DataFrame:
    if n > len(df):
        raise ValueError(f"요청 {n:,}개 > 사용 가능한 {len(df):,}개")

    rng = np.random.default_rng(seed)
    selected_indices = []
    if prefer_balanced:
        per_class = n // 2
        for cls in [0, 1]:
            indices = df.index[df["label"] == cls].to_numpy()
            take = min(per_class, len(indices))
            selected_indices.extend(rng.choice(indices, size=take, replace=False).tolist())
    else:
        # minority(bonafide)를 최대한 보존
        for cls in [0, 1]:
            indices = df.index[df["label"] == cls].to_numpy()
            take = min(n // 2, len(indices))
            selected_indices.extend(rng.choice(indices, size=take, replace=False).tolist())

    remaining = n - len(selected_indices)
    if remaining > 0:
        pool = df.index[~df.index.isin(selected_indices)].to_numpy()
        selected_indices.extend(rng.choice(pool, size=remaining, replace=False).tolist())

    out = df.loc[selected_indices].sample(frac=1.0, random_state=seed).reset_index(drop=True)
    assert len(out) == n
    return out


n_train = cfg.N_TOTAL - cfg.N_VAL
train_df = sample_rows(train_all, n_train, cfg.SEED, prefer_balanced=False)
val_df = sample_rows(dev_all, cfg.N_VAL, cfg.SEED + 1, prefer_balanced=True)

manifest = pd.concat([train_df, val_df], ignore_index=True)
manifest.to_csv(MANIFEST_PATH, index=False, encoding="utf-8")

assert len(train_df) + len(val_df) == cfg.N_TOTAL == 20_000
assert set(train_df["utt_id"]).isdisjoint(set(val_df["utt_id"]))

print("학습 분포:\n", train_df["label"].value_counts().sort_index())
print("검증 분포:\n", val_df["label"].value_counts().sort_index())
print("합계:", len(train_df) + len(val_df))
print("manifest 저장:", MANIFEST_PATH)

## 4. 오디오 Dataset과 통신환경 augmentation

실제 평가에는 전화채널이 일부 포함되므로 학습에서 다음을 양 클래스에 동일하게 적용합니다.

- 8 kHz 다운샘플 후 16 kHz 복원
- gain 변화
- 작은 Gaussian noise
- clipping

augmentation을 real에만 또는 fake에만 적용하면 모델이 생성 흔적 대신 augmentation 흔적을
학습할 수 있으므로 반드시 양쪽에 동일한 확률로 적용합니다.

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(cfg.SEED)


def load_audio_file(path: str, target_sr: int = 16_000) -> torch.Tensor:
    audio, sr = sf.read(path, dtype="float32", always_2d=True)
    wave = torch.from_numpy(audio.T).mean(dim=0)
    if sr != target_sr:
        wave = AF.resample(wave, sr, target_sr)
    wave = torch.nan_to_num(wave).float()
    return wave


def crop_or_repeat(wave: torch.Tensor, length: int, training: bool) -> torch.Tensor:
    n = wave.numel()
    if n == 0:
        return torch.zeros(length)
    if n < length:
        repeats = int(np.ceil(length / n))
        return wave.repeat(repeats)[:length]
    if n == length:
        return wave
    if training:
        start = random.randint(0, n - length)
    else:
        start = (n - length) // 2
    return wave[start:start + length]


def augment_wave(wave: torch.Tensor, sr: int) -> torch.Tensor:
    if random.random() < 0.30:
        wave = AF.resample(AF.resample(wave, sr, 8_000), 8_000, sr)
    if random.random() < 0.35:
        gain = 10 ** (random.uniform(-6.0, 4.0) / 20.0)
        wave = wave * gain
    if random.random() < 0.30:
        signal_rms = wave.square().mean().sqrt().clamp_min(1e-5)
        snr_db = random.uniform(20.0, 40.0)
        noise_rms = signal_rms / (10 ** (snr_db / 20.0))
        wave = wave + torch.randn_like(wave) * noise_rms
    if random.random() < 0.15:
        threshold = random.uniform(0.5, 0.95)
        wave = wave.clamp(-threshold, threshold) / threshold
    return wave.clamp(-1.0, 1.0)


def normalize_for_xlsr(wave: torch.Tensor) -> torch.Tensor:
    return (wave - wave.mean()) / wave.std(unbiased=False).clamp_min(1e-5)


class ASVSpoofDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, training: bool):
        self.frame = frame.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        wave = load_audio_file(row.path, cfg.SAMPLE_RATE)
        wave = crop_or_repeat(wave, cfg.MAX_SAMPLES, self.training)
        if self.training:
            wave = augment_wave(wave, cfg.SAMPLE_RATE)
        wave = normalize_for_xlsr(wave)
        return wave, torch.tensor(float(row.label), dtype=torch.float32)


train_dataset = ASVSpoofDataset(train_df, training=True)
val_dataset = ASVSpoofDataset(val_df, training=False)

class_counts = train_df["label"].value_counts().to_dict()
sample_weights = train_df["label"].map(lambda x: 1.0 / class_counts[int(x)]).to_numpy()
sampler = WeightedRandomSampler(
    torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_dataset),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    sampler=sampler,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
    persistent_workers=cfg.NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=max(1, cfg.BATCH_SIZE * 2),
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
    persistent_workers=cfg.NUM_WORKERS > 0,
)

wave0, label0 = train_dataset[0]
print(wave0.shape, label0, wave0.mean().item(), wave0.std(unbiased=False).item())

## 5. XLS-R + Attentive Statistics Pooling 모델

300M backbone 전체를 학습하면 Colab T4에서 느리고 메모리 사용량이 큽니다. 전체 backbone을
먼저 freeze한 뒤 마지막 `UNFREEZE_LAST_N`개 layer와 pooling/classifier만 학습합니다.
체크포인트에는 XLS-R 설정과 전체 state dict를 함께 저장하므로 DACON의 오프라인 서버에서
Hugging Face 다운로드 없이 복원됩니다.

In [ ]:
class AttentiveStatsPool(nn.Module):
    def __init__(self, dim: int, attention_dim: int = 128):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Conv1d(dim, attention_dim, kernel_size=1),
            nn.Tanh(),
            nn.Conv1d(attention_dim, 1, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [batch, time, channel]
        x = x.transpose(1, 2)
        alpha = torch.softmax(self.attention(x), dim=-1)
        mean = torch.sum(alpha * x, dim=-1)
        second = torch.sum(alpha * x.square(), dim=-1)
        std = (second - mean.square()).clamp_min(1e-5).sqrt()
        return torch.cat([mean, std], dim=1)


class XLSRAntiSpoof(nn.Module):
    def __init__(self, backbone: Wav2Vec2Model):
        super().__init__()
        self.backbone = backbone
        hidden = backbone.config.hidden_size
        self.pre_pool = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.10),
        )
        self.pool = AttentiveStatsPool(256, 128)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 1),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        hidden = self.backbone(input_values=input_values).last_hidden_state
        hidden = self.pre_pool(hidden)
        pooled = self.pool(hidden)
        return self.classifier(pooled).squeeze(-1)


def freeze_for_colab(backbone: Wav2Vec2Model, unfreeze_last_n: int):
    for param in backbone.parameters():
        param.requires_grad = False
    if unfreeze_last_n > 0:
        for layer in backbone.encoder.layers[-unfreeze_last_n:]:
            for param in layer.parameters():
                param.requires_grad = True
        for param in backbone.encoder.layer_norm.parameters():
            param.requires_grad = True


device = torch.device("cuda")
backbone = AutoModel.from_pretrained(cfg.MODEL_ID)
freeze_for_colab(backbone, cfg.UNFREEZE_LAST_N)
backbone.gradient_checkpointing_enable()
model = XLSRAntiSpoof(backbone).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"전체 파라미터: {total_params / 1e6:.1f}M")
print(f"학습 파라미터: {trainable_params / 1e6:.1f}M")

## 6. DACON EER 및 학습

`FAKE=1`로 두고 DACON에 공개된 계산식과 동일하게 EER을 구합니다.
best 기준은 검증 BCE가 아니라 검증 EER입니다.

In [ ]:
def dacon_eer(y_true: np.ndarray, y_score: np.ndarray) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1, drop_intermediate=False)
    fnr = 1.0 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    return float((fpr[idx] + fnr[idx]) / 2.0)


backbone_params = [
    p for name, p in model.named_parameters()
    if name.startswith("backbone.") and p.requires_grad
]
head_params = [
    p for name, p in model.named_parameters()
    if not name.startswith("backbone.") and p.requires_grad
]

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": cfg.LR_BACKBONE},
        {"params": head_params, "lr": cfg.LR_HEAD},
    ],
    weight_decay=cfg.WEIGHT_DECAY,
)

updates_per_epoch = int(np.ceil(len(train_loader) / cfg.GRAD_ACCUM_STEPS))
total_updates = updates_per_epoch * cfg.EPOCHS
warmup_updates = int(total_updates * cfg.WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_updates, total_updates)
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler("cuda", enabled=True)


@torch.inference_mode()
def evaluate(model: nn.Module, loader: DataLoader):
    model.eval()
    all_labels, all_scores = [], []
    total_loss = 0.0
    for waves, labels in tqdm(loader, desc="validation", leave=False):
        waves = waves.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(waves)
            loss = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        all_labels.append(labels.cpu().numpy())
        all_scores.append(torch.sigmoid(logits).float().cpu().numpy())
    y_true = np.concatenate(all_labels)
    y_score = np.concatenate(all_scores)
    return total_loss / len(loader.dataset), dacon_eer(y_true, y_score)


def save_best_checkpoint(model: XLSRAntiSpoof, epoch: int, val_eer: float, val_loss: float):
    checkpoint = {
        "format_version": 1,
        "architecture": "XLSRAntiSpoof-ASP-v1",
        "model_id": cfg.MODEL_ID,
        "model_state": model.state_dict(),
        "backbone_config": model.backbone.config.to_dict(),
        "sample_rate": cfg.SAMPLE_RATE,
        "segment_samples": cfg.MAX_SAMPLES,
        "epoch": epoch,
        "val_eer": val_eer,
        "val_loss": val_loss,
        "training_config": asdict(cfg),
        "manifest_name": MANIFEST_PATH.name,
    }
    torch.save(checkpoint, BEST_PATH)


best_eer = float("inf")
history = []
optimizer.zero_grad(set_to_none=True)

for epoch in range(1, cfg.EPOCHS + 1):
    model.train()
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"epoch {epoch}/{cfg.EPOCHS}")

    for step, (waves, labels) in enumerate(progress, start=1):
        waves = waves.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(waves)
            loss = criterion(logits, labels) / cfg.GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * cfg.GRAD_ACCUM_STEPS

        should_update = step % cfg.GRAD_ACCUM_STEPS == 0 or step == len(train_loader)
        if should_update:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        if step % 100 == 0:
            progress.set_postfix(loss=f"{running_loss / step:.4f}")

    val_loss, val_eer = evaluate(model, val_loader)
    row = {
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "val_loss": val_loss,
        "val_eer": val_eer,
    }
    history.append(row)
    print(row)

    if val_eer < best_eer:
        best_eer = val_eer
        save_best_checkpoint(model, epoch, val_eer, val_loss)
        print(f"best.pt 저장: {BEST_PATH} (EER={best_eer:.6f})")

pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print("최종 best EER:", best_eer)
print("best.pt 크기(GB):", BEST_PATH.stat().st_size / 1024**3)

## 7. `best.pt` 복원 검사

인터넷을 사용하지 않고 config와 state dict만으로 복원되는지 확인합니다.

In [ ]:
del model, backbone, optimizer, scheduler, scaler, backbone_params, head_params
torch.cuda.empty_cache()

try:
    checkpoint = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
except TypeError:
    checkpoint = torch.load(BEST_PATH, map_location="cpu")

offline_config = Wav2Vec2Config.from_dict(checkpoint["backbone_config"])
offline_model = XLSRAntiSpoof(Wav2Vec2Model(offline_config))
missing, unexpected = offline_model.load_state_dict(checkpoint["model_state"], strict=False)
assert not missing and not unexpected, (missing, unexpected)
offline_model.eval().to(device)

with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
    test_logit = offline_model(wave0[None].to(device))
print("오프라인 복원 성공, 예측 확률:", torch.sigmoid(test_logit).item())

del offline_model, checkpoint
torch.cuda.empty_cache()

## 8. PANNs CNN14 체크포인트 준비

PANNs는 AudioSet의 `Speech`, `Singing`, `Music` 계열 클래스를 이용해 음성/음악 존재 확률을
만듭니다. 첫 실행 시 약 300 MB 체크포인트를 다운로드한 뒤 제출용 `model/`에 포함합니다.
DACON 서버에서는 인터넷 다운로드를 하지 않습니다.

In [ ]:
from panns_inference import AudioTagging

PANNS_DRIVE_PATH = OUTPUT_DIR / "Cnn14_mAP=0.431.pth"

if not PANNS_DRIVE_PATH.exists():
    print("PANNs 체크포인트 다운로드/초기화 중...")
    panns_temp = AudioTagging(checkpoint_path=None, device="cpu")
    default_panns_path = Path.home() / "panns_data" / "Cnn14_mAP=0.431.pth"
    if not default_panns_path.exists():
        candidates = list((Path.home() / "panns_data").glob("*Cnn14*.pth"))
        if not candidates:
            raise FileNotFoundError("PANNs가 다운로드한 Cnn14 체크포인트를 찾지 못했습니다.")
        default_panns_path = candidates[0]
    shutil.copy2(default_panns_path, PANNS_DRIVE_PATH)
    del panns_temp

print("PANNs checkpoint:", PANNS_DRIVE_PATH)
print("PANNs size(MB):", PANNS_DRIVE_PATH.stat().st_size / 1024**2)

## 9. DACON 오프라인 `script.py`와 `requirements.txt` 생성

추론 코드의 특징:

- `data/test`와 `data/sample_submission.csv`를 상대경로로 읽음
- MP3/WAV/FLAC를 torchaudio로 읽고, 실패 시 시스템 ffmpeg 사용
- 파일마다 독립적으로 최대 5개 구간을 추론해 mean/max logit 결합
- 다른 평가 파일의 평균·순위·통계를 사용하지 않음
- 출력 확률을 반올림하지 않고 `1e-6 ~ 1-1e-6`로만 제한
- 반드시 `output/submission.csv`로 저장

In [ ]:
INFERENCE_SCRIPT = r'''from pathlib import Path
import math
import subprocess
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio
import torchaudio.functional as AF
from transformers import Wav2Vec2Config, Wav2Vec2Model
from panns_inference import AudioTagging, labels as PANNS_LABELS


BASE_DIR = Path(__file__).resolve().parent
MODEL_DIR = BASE_DIR / "model"
DATA_DIR = BASE_DIR / "data"
TEST_DIR = DATA_DIR / "test"
OUTPUT_DIR = BASE_DIR / "output"
BEST_PATH = MODEL_DIR / "best.pt"
PANNS_PATH = MODEL_DIR / "Cnn14_mAP=0.431.pth"

SAMPLE_RATE = 16_000
PANNS_SAMPLE_RATE = 32_000
SEGMENT_SAMPLES = 64_000
MAX_XLSR_CROPS = 5
EPS = 1e-6
AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac"}


class AttentiveStatsPool(nn.Module):
    def __init__(self, dim: int, attention_dim: int = 128):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Conv1d(dim, attention_dim, kernel_size=1),
            nn.Tanh(),
            nn.Conv1d(attention_dim, 1, kernel_size=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.transpose(1, 2)
        alpha = torch.softmax(self.attention(x), dim=-1)
        mean = torch.sum(alpha * x, dim=-1)
        second = torch.sum(alpha * x.square(), dim=-1)
        std = (second - mean.square()).clamp_min(1e-5).sqrt()
        return torch.cat([mean, std], dim=1)


class XLSRAntiSpoof(nn.Module):
    def __init__(self, backbone: Wav2Vec2Model):
        super().__init__()
        self.backbone = backbone
        hidden = backbone.config.hidden_size
        self.pre_pool = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.10),
        )
        self.pool = AttentiveStatsPool(256, 128)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 1),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        hidden = self.backbone(input_values=input_values).last_hidden_state
        hidden = self.pre_pool(hidden)
        return self.classifier(self.pool(hidden)).squeeze(-1)


def load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def load_audio(path: Path) -> torch.Tensor:
    try:
        wave, sr = torchaudio.load(str(path))
        wave = wave.float().mean(dim=0)
        if sr != SAMPLE_RATE:
            wave = AF.resample(wave, sr, SAMPLE_RATE)
    except Exception:
        command = [
            "ffmpeg", "-v", "error", "-i", str(path),
            "-f", "f32le", "-ac", "1", "-ar", str(SAMPLE_RATE), "pipe:1"
        ]
        result = subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        wave = torch.from_numpy(np.frombuffer(result.stdout, dtype=np.float32).copy())
    if wave.numel() == 0:
        raise RuntimeError(f"빈 오디오입니다: {path}")
    return torch.nan_to_num(wave).clamp(-1.0, 1.0)


def repeat_to_length(wave: torch.Tensor, length: int) -> torch.Tensor:
    if wave.numel() >= length:
        return wave[:length]
    repeats = int(math.ceil(length / max(1, wave.numel())))
    return wave.repeat(repeats)[:length]


def uniform_xlsr_crops(wave: torch.Tensor) -> torch.Tensor:
    n = wave.numel()
    if n <= SEGMENT_SAMPLES:
        crops = [repeat_to_length(wave, SEGMENT_SAMPLES)]
    else:
        crop_count = min(MAX_XLSR_CROPS, int(math.ceil(n / SEGMENT_SAMPLES)))
        starts = np.linspace(0, n - SEGMENT_SAMPLES, crop_count).round().astype(int)
        crops = [wave[s:s + SEGMENT_SAMPLES] for s in starts]
    normalized = []
    for crop in crops:
        crop = (crop - crop.mean()) / crop.std(unbiased=False).clamp_min(1e-5)
        normalized.append(crop)
    return torch.stack(normalized)


def uniform_panns_crops(wave_32k: torch.Tensor, crop_seconds: int = 10, max_crops: int = 6):
    crop_length = PANNS_SAMPLE_RATE * crop_seconds
    n = wave_32k.numel()
    if n <= crop_length:
        return [wave_32k]
    count = min(max_crops, int(math.ceil(n / crop_length)))
    starts = np.linspace(0, n - crop_length, count).round().astype(int)
    return [wave_32k[s:s + crop_length] for s in starts]


def aggregate_xlsr(model, crops, device):
    logits = []
    with torch.inference_mode():
        for start in range(0, len(crops), 4):
            batch = crops[start:start + 4].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits.append(model(batch).float().cpu())
    logits = torch.cat(logits)
    # 국소 Fake와 파일 전체 정보를 함께 반영. 파일 내부에서만 집계한다.
    combined_logit = 0.5 * logits.mean() + 0.5 * logits.max()
    return float(torch.sigmoid(combined_logit).item())


VOICE_LABEL_NAMES = [
    "Speech", "Male speech, man speaking", "Female speech, woman speaking",
    "Child speech, kid speaking", "Conversation", "Narration, monologue",
    "Singing", "Male singing", "Female singing", "Child singing", "Choir",
    "Vocal music",
]
MUSIC_LABEL_NAMES = [
    "Music", "Musical instrument", "Orchestra", "Background music",
    "Electronic music", "Independent music",
]


def label_indices(names):
    label_list = list(PANNS_LABELS)
    return [label_list.index(name) for name in names if name in label_list]


VOICE_INDICES = label_indices(VOICE_LABEL_NAMES)
MUSIC_INDICES = label_indices(MUSIC_LABEL_NAMES)


def panns_presence(panns, wave_16k):
    wave_32k = AF.resample(wave_16k, SAMPLE_RATE, PANNS_SAMPLE_RATE)
    crops = uniform_panns_crops(wave_32k)
    outputs = []
    for crop in crops:
        clipwise, _ = panns.inference(crop[None].cpu().numpy())
        outputs.append(clipwise[0])
    output = np.stack(outputs)
    voice_prob = float(output[:, VOICE_INDICES].max()) if VOICE_INDICES else 0.5
    music_prob = float(output[:, MUSIC_INDICES].max()) if MUSIC_INDICES else 0.5
    return voice_prob, music_prob


def clamp_probability(value):
    return float(np.clip(value, EPS, 1.0 - EPS))


def main():
    started = time.time()
    if not BEST_PATH.exists() or not PANNS_PATH.exists():
        raise FileNotFoundError("model/best.pt 또는 PANNs 체크포인트가 없습니다.")
    if not TEST_DIR.exists():
        raise FileNotFoundError(f"평가 데이터 폴더가 없습니다: {TEST_DIR}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = load_checkpoint(BEST_PATH)
    config = Wav2Vec2Config.from_dict(checkpoint["backbone_config"])
    model = XLSRAntiSpoof(Wav2Vec2Model(config))
    model.load_state_dict(checkpoint["model_state"], strict=True)
    model.eval().to(device)

    panns = AudioTagging(checkpoint_path=str(PANNS_PATH), device=str(device))

    audio_paths = [
        p for p in TEST_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS
    ]
    audio_map = {p.stem: p for p in audio_paths}
    if not audio_map:
        raise RuntimeError(f"평가 오디오가 없습니다: {TEST_DIR}")

    sample_path = DATA_DIR / "sample_submission.csv"
    if sample_path.exists():
        submission = pd.read_csv(sample_path, dtype={"ID": str})
    else:
        submission = pd.DataFrame({"ID": sorted(audio_map)})

    required_columns = [
        "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
        "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
    ]
    for col in required_columns:
        if col not in submission.columns:
            submission[col] = 0.0

    predictions = []
    for row_index, sample_id in enumerate(submission["ID"].astype(str), start=1):
        audio_path = audio_map.get(sample_id) or audio_map.get(Path(sample_id).stem)
        if audio_path is None:
            raise FileNotFoundError(f"ID에 해당하는 오디오가 없습니다: {sample_id}")

        wave = load_audio(audio_path)
        anti_spoof_prob = aggregate_xlsr(model, uniform_xlsr_crops(wave), device)
        voice_present, music_present = panns_presence(panns, wave)

        # ASVspoof-only 모델의 명시적 fallback.
        # 음악 Fake 데이터로 specialist를 학습한 뒤 이 값만 교체하면 된다.
        voice_fake = anti_spoof_prob
        music_fake = anti_spoof_prob
        file_fake = anti_spoof_prob

        predictions.append([
            clamp_probability(file_fake),
            clamp_probability(voice_fake),
            clamp_probability(music_fake),
            clamp_probability(voice_present),
            clamp_probability(music_present),
        ])

        if row_index % 50 == 0 or row_index == len(submission):
            print(f"[{row_index}/{len(submission)}] elapsed={time.time() - started:.1f}s")

    submission.loc[:, required_columns] = np.asarray(predictions, dtype=np.float64)
    submission = submission[["ID"] + required_columns]
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / "submission.csv"
    submission.to_csv(output_path, index=False, encoding="utf-8")
    print(f"saved: {output_path}, rows={len(submission)}, total={time.time() - started:.1f}s")


if __name__ == "__main__":
    main()
'''

REQUIREMENTS_TEXT = '''# DACON 기본 CUDA 빌드(torch==2.7.1+cu128, torchaudio==2.7.1+cu128)는
# 평가 서버에 사전 설치되므로 재설치하지 않습니다.
numpy==1.26.4
pandas==2.0.3
transformers==4.57.6
accelerate==1.9.0
librosa==0.10.2.post1
soundfile==0.12.1
soxr==0.5.0.post1
panns-inference==0.1.1
'''

SUBMISSION_DIR = OUTPUT_DIR / "submit_build"
MODEL_SUBDIR = SUBMISSION_DIR / "model"
MODEL_SUBDIR.mkdir(parents=True, exist_ok=True)

(SUBMISSION_DIR / "script.py").write_text(INFERENCE_SCRIPT, encoding="utf-8")
(SUBMISSION_DIR / "requirements.txt").write_text(REQUIREMENTS_TEXT, encoding="utf-8")
shutil.copy2(BEST_PATH, MODEL_SUBDIR / "best.pt")
shutil.copy2(PANNS_DRIVE_PATH, MODEL_SUBDIR / "Cnn14_mAP=0.431.pth")

print("생성 파일:")
for path in sorted(SUBMISSION_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(SUBMISSION_DIR), f"{path.stat().st_size / 1024**2:.1f} MB")

## 10. `submit.zip` 생성 및 구조 검증

zip 내부 최상위에는 추가 폴더 없이 아래 세 항목이 바로 있어야 합니다.

```text
submit.zip
├── model/
│   ├── best.pt
│   └── Cnn14_mAP=0.431.pth
├── script.py
└── requirements.txt
```

In [ ]:
SUBMIT_ZIP = OUTPUT_DIR / "submit.zip"

with zipfile.ZipFile(
    SUBMIT_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=4,
    allowZip64=True,
) as archive:
    for path in sorted(SUBMISSION_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(SUBMISSION_DIR).as_posix())

with zipfile.ZipFile(SUBMIT_ZIP, "r") as archive:
    names = archive.namelist()
    bad_names = [name for name in names if name.startswith("submit_build/") or name.startswith("submit/")]
    assert not bad_names, bad_names
    assert "script.py" in names
    assert "requirements.txt" in names
    assert "model/best.pt" in names
    assert "model/Cnn14_mAP=0.431.pth" in names
    corrupt = archive.testzip()
    assert corrupt is None, f"손상된 zip 항목: {corrupt}"

zip_size_gb = SUBMIT_ZIP.stat().st_size / 1024**3
print("submit.zip:", SUBMIT_ZIP)
print(f"압축 크기: {zip_size_gb:.2f} GB")
assert zip_size_gb < 10.0, "DACON 압축 파일 10GB 제한을 초과했습니다."
print("ZIP 내부 구조:")
print("\n".join(names))

## 11. 선택 사항: DACON 더미 데이터로 smoke test

DACON에서 받은 `open.zip`을 Google Drive에 압축 해제한 후 `OPEN_ROOT`를 수정하고
`RUN_SMOKE_TEST=True`로 바꾸면, 별도 `/content/dacon_smoke_test`에서 제출 코드를 실행합니다.
원본 open 폴더나 submit.zip은 수정하지 않습니다.

In [ ]:
RUN_SMOKE_TEST = False
OPEN_ROOT = Path("/content/drive/MyDrive/dacon_deepvoice/open")  # data/가 들어 있는 폴더

if RUN_SMOKE_TEST:
    assert (OPEN_ROOT / "data" / "test").exists(), OPEN_ROOT
    smoke_root = Path("/content/dacon_smoke_test")
    if smoke_root.exists():
        shutil.rmtree(smoke_root)
    smoke_root.mkdir(parents=True)
    shutil.copytree(OPEN_ROOT / "data", smoke_root / "data")
    shutil.copytree(MODEL_SUBDIR, smoke_root / "model")
    shutil.copy2(SUBMISSION_DIR / "script.py", smoke_root / "script.py")
    shutil.copy2(SUBMISSION_DIR / "requirements.txt", smoke_root / "requirements.txt")

    result = subprocess.run(
        ["python", "script.py"],
        cwd=smoke_root,
        text=True,
        capture_output=True,
        timeout=3600,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("smoke test 실패")

    smoke_submission = pd.read_csv(smoke_root / "output" / "submission.csv")
    display(smoke_submission)
    assert list(smoke_submission.columns) == [
        "ID", "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
        "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
    ]
    assert smoke_submission.iloc[:, 1:].applymap(lambda x: 0.0 <= x <= 1.0).all().all()
    print("smoke test 통과")
else:
    print("smoke test를 생략했습니다. 제출 전 RUN_SMOKE_TEST=True 실행을 권장합니다.")

## 완료 후 생성되는 파일

Google Drive의 `MyDrive/dacon_deepvoice/` 아래에 다음 파일이 남습니다.

- `best.pt`: 검증 EER 최저 모델
- `asvspoof_selected_20000.csv`: 실제 사용한 정확히 20,000개 manifest
- `training_history.csv`: epoch별 loss/EER
- `Cnn14_mAP=0.431.pth`: 음성/음악 존재 탐지 PANNs 가중치
- `submit.zip`: DACON 리더보드 제출 파일

리더보드 제출 전 확인사항:

1. 더미 데이터 smoke test 통과
2. zip 최상위에 `model/`, `script.py`, `requirements.txt`만 존재
3. `output/submission.csv`가 6개 컬럼(ID + 확률 5개)으로 생성
4. 1,200개 실제 평가에서 60분 이내인지 확인
5. ASVspoof2019, XLS-R, PANNs의 출처와 라이선스를 2차 평가 자료용으로 기록

음악 Fake 점수를 개선하려면 다음 버전에서 FakeMusicCaps/SONICS로 학습한
SpecTTTra 또는 WPT-MERT branch를 추가하고 `music_fake` 및 `file_fake` 결합부를 교체하세요.